# Does the Gaussian support converge?

Treat `frac=2` as the prespecified reference and test whether truncating the Gaussian at smaller supports changes its single environmental magnitude or direction.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import esp_pv_tools as ept
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}


In [ ]:
data = ept.load_cache()
gaussian = data[data.pv_surface_method.eq("esp_gaussian")].copy()
paired = ept.paired_to(gaussian)
paired = paired[paired.method.ne("esp_gaussian_2")].copy()
paired["magnitude_ratio"] = paired.PV_grad_mag / paired.reference_PV_grad_mag
paired["direction_change"] = np.abs((paired.PV_grad_theta-paired.reference_PV_grad_theta+180)%360-180)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
sns.violinplot(data=paired, x="method", y="magnitude_ratio", hue="Cyc", split=True, cut=0, palette=palette, ax=axes[0])
sns.ecdfplot(data=paired, x="direction_change", hue="method", ax=axes[1])
axes[0].axhline(1, color="k", ls="--"); axes[0].set(yscale="log", xlabel="", ylabel="Magnitude / frac=2", title="Magnitude convergence")
axes[1].set(xlim=(0,90), xlabel="Absolute direction change (degrees)", title="Direction convergence")
axes[0].tick_params(axis="x", rotation=20); plt.show()

In [ ]:
summary = paired.groupby(["Cyc","method"]).agg(
    median_ratio=("magnitude_ratio","median"), p90_direction_change=("direction_change",lambda x:x.quantile(.9)),
    median_effective_cells=("PV_weight_effective_n","median")).reset_index()
fig, axes = plt.subplots(1,2,figsize=(12,4.3),constrained_layout=True)
sns.pointplot(data=summary,x="method",y="median_ratio",hue="Cyc",palette=palette,ax=axes[0])
sns.pointplot(data=summary,x="method",y="p90_direction_change",hue="Cyc",palette=palette,ax=axes[1])
for ax in axes: ax.tick_params(axis="x",rotation=20); ax.set_xlabel("")
axes[0].axhline(1,color="k",ls="--"); axes[0].set_title("Median magnitude ratio")
axes[1].set_title("90th-percentile direction change"); plt.show()